In [1]:
# ==========================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E CONFIGURAÇÕES
# ==========================================
import math
import random
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# ==========================================
# 2. LOCALIZAÇÃO DE DIRETÓRIOS E ARQUIVOS
# ==========================================
current_path = Path.cwd().resolve()
project_root = next(
    (
        parent
        for parent in (current_path, *current_path.parents)
        if (parent / "data" / "data_ibge_2022").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        f"Não encontrei a pasta data/data_ibge_2022 a partir de {current_path}. "
        "Abra o notebook dentro do projeto e confira se os dados foram baixados."
    )

data_root = project_root / "data" / "data_ibge_2022"
data_path = data_root / "censo_2022_35_SP_domicilios" / "35"

# Caminhos dos arquivos CSV de microdados controlados
file_01 = data_path / "Domicilios_35_controlado.csv"
file_04 = data_path / "Pessoas_35_controlado.csv"

for caminho in (file_01, file_04):
    if not caminho.is_file():
        raise FileNotFoundError(f"Arquivo de entrada não encontrado: {caminho}")

# ==========================================
# 3. FUNÇÕES UTILITÁRIAS
# ==========================================
def ler_csv_com_amostragem(caminho_arquivo, tamanho_amostra, selecao_colunas, random_state=42):
    """
    Lê um arquivo CSV grande e retorna uma amostra aleatória de linhas.
    """
    caminho_arquivo = Path(caminho_arquivo)
    if not caminho_arquivo.is_file():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho_arquivo}")
    if isinstance(tamanho_amostra, bool) or not isinstance(tamanho_amostra, int):
        raise TypeError("tamanho_amostra deve ser um inteiro.")
    if tamanho_amostra < 0:
        raise ValueError("tamanho_amostra não pode ser negativo.")

    with caminho_arquivo.open("r", encoding="ASCII") as f:
        total_linhas = sum(1 for _ in f) - 1  # desconsidera o cabeçalho
    if total_linhas < 0:
        raise ValueError(f"O arquivo não contém cabeçalho: {caminho_arquivo}")
    if tamanho_amostra > total_linhas:
        raise ValueError(
            f"A amostra solicitada ({tamanho_amostra}) é maior que o total de linhas ({total_linhas})."
        )

    linhas_amostradas = set(
        random.Random(random_state).sample(range(1, total_linhas + 1), tamanho_amostra)
    )
    df_amostra = pd.read_csv(
        caminho_arquivo,
        skiprows=lambda indice: indice > 0 and indice not in linhas_amostradas,
        usecols=selecao_colunas,
        sep=";",
        encoding="ASCII",
    )
    if len(df_amostra) != tamanho_amostra:
        raise ValueError(
            f"Esperadas {tamanho_amostra} linhas amostradas, mas foram lidas {len(df_amostra)}."
        )
    return df_amostra


def imprimir_matriz(lista, num_colunas=None):
    """
    Recebe uma lista e imprime seus elementos em formato de matriz com alinhamento.
    """
    total = len(lista)
    if total == 0:
        print("Lista vazia.")
        return

    if num_colunas is None:
        num_colunas = math.ceil(math.sqrt(total))

    if num_colunas <= 0:
        raise ValueError("O número de colunas deve ser positivo.")

    linhas = [lista[i:i + num_colunas] for i in range(0, total, num_colunas)]
    larguras = [0] * num_colunas

    for linha in linhas:
        for idx, item in enumerate(linha):
            larguras[idx] = max(larguras[idx], len(str(item)))

    for linha in linhas:
        linha_formatada = [
            str(item).ljust(larguras[idx]) for idx, item in enumerate(linha)
        ]
        print("  ".join(linha_formatada))


# ==========================================
# 4. MAPEAMENTO E SELEÇÃO DE COLUNAS
# ==========================================
mapa_domicilios = {
    "D0020": "uf",
    "D0080": "municipio",
    "D0090": "area_ponderacao",
    "D0100": "controle",
    "D0111": "peso_amostral",
    "D0120": "situacao_setor",
    "D0130": "especie_unidade",
    "D0140": "situacao_domicilio",
    "D0150": "num_moradores",
    "D0160": "num_criancas",
    "D0170": "sexo_responsavel",
    "D0180": "idade_responsavel_cat",
    "D0220": "num_comodos",
    "D0230": "num_dormitorios",
    "D0300": "agua_canalizada",
    "D0330": "acesso_internet",
    "D0350": "rendimento_domiciliar",
    "D0360": "rendimento_per_capita",
}

mapa_pessoas = {
    "P0020": "uf",
    "P0080": "municipio",
    "P0090": "area_ponderacao",
    "P0100": "controle",
    "P0111": "peso_amostral",
    "P0120": "situacao_setor",
    "P0130": "especie_unidade",
    "P0140": "situacao_domicilio",
    "P0180": "idade_cat",
    "P0210": "cor_raca",
    "P0220": "indigena",
    "P0480": "local_nascimento",
    "P0490": "uf_nascimento",
    "P0520": "nacionalidade",
    "P0640": "alfabetizacao",
    "P0650": "frequencia_escolar",
    "P0660": "curso_frequentado",
    "P0670": "ano_curso",
    "P0690": "outra_graduacao",
    "P0700": "curso_anterior_elevado",
    "P0760": "nivel_instrucao",
    "P0790": "anos_estudo",
    "P0840": "trabalho_remunerado",
    "P0860": "bico_ocasional",
    "P0900": "qtd_trabalhos",
    "P0990": "posicao_ocupacao",
    "P1000": "carteira_assinada",
    "P1010": "registro_cnpj",
    "P1080": "rendimento_trabalhos",
    "P1110": "rendimento_todas_fontes",
    "P1170": "meio_transporte_trabalho",
    "P1180": "tempo_deslocamento_cat",
    "P1190": "tempo_deslocamento_min",
    "P1210": "informante",
}

# Leitura dos dados tabulares filtrando apenas São Paulo (Código Município: 3550308)
df_ibge_2022_domicilios = pd.read_csv(
    file_01, usecols=mapa_domicilios.keys(), sep=";", encoding="ASCII", low_memory=False
)
df_ibge_2022_domicilios = df_ibge_2022_domicilios[df_ibge_2022_domicilios["D0080"] == 3550308]
df_ibge_2022_domicilios.rename(columns=mapa_domicilios, inplace=True)

df_ibge_2022_pessoas = ler_csv_com_amostragem(
    file_04, tamanho_amostra=10000, selecao_colunas=mapa_pessoas.keys()
)
df_ibge_2022_pessoas = df_ibge_2022_pessoas[df_ibge_2022_pessoas["P0080"] == 3550308]
df_ibge_2022_pessoas.rename(columns=mapa_pessoas, inplace=True)


# ==========================================
# 5. PROCESSAMENTO ESPACIAL E GEOGRÁFICO
# ==========================================
# Arquivos geográficos
file_05 = data_root / "SP_bairros_CD2022.gpkg"
file_06_sub = data_root / "SP_subdistritos_CD2022.gpkg"
file_07_setores = data_root / "SP_setores_CD2022.gpkg"

gdf_setores = gpd.read_file(file_07_setores)

# Filtrando setores apenas para o município de São Paulo (CD_MUN: 3550308)
gdf_setores_sp = gdf_setores[gdf_setores["CD_MUN"].astype(str).str.strip() == "3550308"].copy()
gdf_setores_sp["CD_SETOR"] = gdf_setores_sp["CD_SETOR"].astype(str).str.strip()

# Carregando tabela de composição (De-Para entre Setor Censitário e Área de Ponderação)
file_aponds_txt = data_root / "APONDs_SP.txt"
file_aponds_excel = data_root / "Composição das Áreas de Ponderação.xlsx"

if file_aponds_txt.is_file():
    df_comp = pd.read_csv(file_aponds_txt, sep=";", dtype=str)
elif file_aponds_excel.is_file():
    df_comp = pd.read_excel(file_aponds_excel, dtype=str)
else:
    raise FileNotFoundError("Arquivo de composição das áreas de ponderação (APONDs_SP.txt ou Excel) não encontrado.")

# Padronizando colunas da tabela de composição para o merge
df_comp.columns = df_comp.columns.str.strip()
if "Setor" in df_comp.columns:
    df_comp.rename(columns={"Setor": "CD_SETOR"}, inplace=True)

df_comp["CD_SETOR"] = df_comp["CD_SETOR"].str.strip()

# Identificando a coluna de Área de Ponderação no arquivo de composição
coluna_apond = next((c for c in df_comp.columns if "ponderação" in c.lower() or "apond" in c.lower() or "cd_aponde" in c.lower()), "Área de ponderação")

# Realizando o merge dos setores de SP com a respectiva Área de Ponderação ANTES do dissolve
gdf_setores_sp = gdf_setores_sp.merge(
    df_comp[["CD_SETOR", coluna_apond]], 
    on="CD_SETOR", 
    how="inner"
)

# Convertendo AREA_KM2 para numérico caso necessário
gdf_setores_sp["AREA_KM2"] = pd.to_numeric(gdf_setores_sp["AREA_KM2"], errors="coerce")

# Dissolvendo os setores geográficos para formar as Áreas de Ponderação oficiais
gdf_areas_pond = gdf_setores_sp.dissolve(
    by=coluna_apond,
    aggfunc={"AREA_KM2": "sum"},
    as_index=False
)

print(f"Processamento concluído com sucesso! Total de Áreas de Ponderação geradas para São Paulo: {len(gdf_areas_pond)}")

KeyError: 'CD_SETOR'